# Data Engineering for ML

## ETL vs ELT

| Approach | Steps | When to Use |
|----------|-------|-------------|
| **ETL** | Extract → Transform → Load | Data warehouse era, structured data |
| **ELT** | Extract → Load → Transform | Cloud data lakes, raw data first |

```
ETL: Source → [Transform] → Warehouse
ELT: Source → Data Lake → [Transform in-place]
```

## The Modern Data Stack

```
Sources           Ingestion       Storage          Transform     Serve
──────────────────────────────────────────────────────────────────────
Databases    →  Kafka/Airbyte →  S3/GCS/ADLS  →  dbt/Spark  →  BI/ML
APIs         →  Fivetran      →  Snowflake     →  Spark SQL  →  FastAPI
App logs     →  Logstash      →  Iceberg/Delta →  Flink      →  Dashboards
```

## Apache Spark

Spark is the standard for **distributed data processing**. Uses a **DAG** (Directed Acyclic Graph) of transformations.

### Key Concepts
- **RDD** (Resilient Distributed Dataset): immutable distributed collection
- **DataFrame**: structured data with named columns (like pandas, but distributed)
- **Lazy evaluation**: transformations are not executed until an action is called
- **Catalyst optimizer**: automatically optimizes query plans

### Spark vs Pandas

| | Pandas | Spark |
|--|--------|-------|
| Size | Fits in RAM | Petabytes |
| Execution | Eager | Lazy |
| API | Simpler | More verbose |
| SQL | partial | Full SQL |
| Streaming | ❌ | ✅ Structured Streaming |

In [1]:
# pip install pyspark

spark_example = '''
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("MLPipeline") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

# Read data
df = spark.read.csv("s3://my-bucket/data/users.csv", header=True, inferSchema=True)

# Transformations (lazy not executed yet)
df_clean = (
    df.filter(F.col("age") > 0)
      .withColumn("age_bucket", F.when(F.col("age") < 30, "young")
                                  .when(F.col("age") < 60, "middle")
                                  .otherwise("senior"))
      .withColumn("name_upper", F.upper(F.col("name")))
      .dropDuplicates(["user_id"])
      .na.fill({"age": 0, "income": 0.0})
)

# Aggregations
agg_df = (
    df_clean.groupBy("age_bucket", "country")
            .agg(
                F.count("*").alias("count"),
                F.avg("income").alias("avg_income"),
                F.percentile_approx("income", 0.5).alias("median_income")
            )
            .orderBy(F.desc("count"))
)

# SQL interface
df_clean.createOrReplaceTempView("users")
spark.sql("""
    SELECT age_bucket, COUNT(*) as cnt, AVG(income) as avg_income
    FROM users
    WHERE income > 0
    GROUP BY age_bucket
    ORDER BY avg_income DESC
""").show()

# ML Pipeline with Spark MLlib
indexer = StringIndexer(inputCol="category", outputCol="category_idx")
assembler = VectorAssembler(
    inputCols=["age", "income", "category_idx"],
    outputCol="features"
)
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")
rf = RandomForestClassifier(featuresCol="scaled_features", labelCol="label", numTrees=100)

pipeline = Pipeline(stages=[indexer, assembler, scaler, rf])
model = pipeline.fit(train_df)
predictions = model.transform(test_df)

# Write back to storage
agg_df.write.mode("overwrite").parquet("s3://my-bucket/output/agg_users/")
agg_df.write.mode("overwrite").partitionBy("country").parquet("s3://output/partitioned/")
'''
print(spark_example)


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

spark = SparkSession.builder     .appName("MLPipeline")     .config("spark.driver.memory", "4g")     .getOrCreate()

# Read data
df = spark.read.csv("s3://my-bucket/data/users.csv", header=True, inferSchema=True)

# Transformations (lazy not executed yet)
df_clean = (
    df.filter(F.col("age") > 0)
      .withColumn("age_bucket", F.when(F.col("age") < 30, "young")
                                  .when(F.col("age") < 60, "middle")
                                  .otherwise("senior"))
      .withColumn("name_upper", F.upper(F.col("name")))
      .dropDuplicates(["user_id"])
      .na.fill({"age": 0, "income": 0.0})
)

# Aggregations
agg_df = (
    df

## Apache Kafka Event Streaming

Kafka is a **distributed event streaming platform** for real-time data pipelines.

### Core Concepts
- **Topic**: named stream of records (like a category/feed)
- **Partition**: topic split for parallelism, ordered within partition
- **Offset**: position of a record within a partition
- **Consumer Group**: multiple consumers share work on a topic
- **Broker**: Kafka server that stores and serves messages

```
Producer → [Topic: predictions] → Partition 0 → Consumer Group A
                                → Partition 1 → Consumer Group A
                                → Partition 2 → Consumer Group B
```

### Delivery Guarantees
- `acks=0`: Fire and forget (fastest, possible loss)
- `acks=1`: Leader acknowledges (default)
- `acks=all`: All replicas acknowledge (slowest, no loss)

In [2]:
# pip install confluent-kafka
# Start Kafka: docker-compose up kafka

kafka_example = '''
from confluent_kafka import Producer, Consumer
import json
import time

# --- PRODUCER (ML model serving results) ---
producer = Producer({
    "bootstrap.servers": "localhost:9092",
    "acks": "all",              # strongest delivery guarantee
    "retries": 3,
    "compression.type": "gzip"
})

def send_prediction(user_id, input_data, prediction, confidence):
    message = {
        "user_id": user_id,
        "input": input_data,
        "prediction": prediction,
        "confidence": confidence,
        "timestamp": time.time()
    }
    producer.produce(
        topic="ml.predictions",
        key=str(user_id).encode(),
        value=json.dumps(message).encode()
    )
    producer.flush()

# --- CONSUMER (downstream processing) ---
consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "prediction-logger",
    "auto.offset.reset": "earliest",   # or "latest"
    "enable.auto.commit": False          # manual commit for exactly-once
})
consumer.subscribe(["ml.predictions"])

try:
    while True:
        msg = consumer.poll(timeout=1.0)
        if msg is None:
            continue
        if msg.error():
            print(f"Error: {msg.error()}")
            continue
        data = json.loads(msg.value())
        # Process: save to DB, update monitoring
        print(f"User {data[\'user_id\']}: {data[\'prediction\']} ({data[\'confidence\']:.2f})")
        consumer.commit()  # mark as processed
finally:
    consumer.close()

# Kafka Streams / Flink for real-time aggregations:
# - Count predictions per model per minute
# - Alert when confidence drops below threshold
# - Join with user profile data in real-time
'''
print(kafka_example)


from confluent_kafka import Producer, Consumer
import json
import time

# --- PRODUCER (ML model serving results) ---
producer = Producer({
    "bootstrap.servers": "localhost:9092",
    "acks": "all",              # strongest delivery guarantee
    "retries": 3,
    "compression.type": "gzip"
})

def send_prediction(user_id, input_data, prediction, confidence):
    message = {
        "user_id": user_id,
        "input": input_data,
        "prediction": prediction,
        "confidence": confidence,
        "timestamp": time.time()
    }
    producer.produce(
        topic="ml.predictions",
        key=str(user_id).encode(),
        value=json.dumps(message).encode()
    )
    producer.flush()

# --- CONSUMER (downstream processing) ---
consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "prediction-logger",
    "auto.offset.reset": "earliest",   # or "latest"
    "enable.auto.commit": False          # manual commit for exactly-once
})
consumer.subscribe(

## dbt Data Build Tool

dbt brings **software engineering practices** to SQL transformations:
- Version control for SQL models
- Automatic documentation
- Built-in testing
- Dependency management

In [3]:
dbt_example = '''
# models/staging/stg_predictions.sql
{{ config(materialized="view") }}

SELECT
    id,
    user_id,
    model_name,
    prediction,
    confidence::FLOAT AS confidence,
    created_at::TIMESTAMP AS created_at
FROM {{ source(\'raw\', \'predictions\') }}
WHERE created_at IS NOT NULL

---

# models/marts/model_performance.sql
{{ config(materialized="table") }}

WITH daily_preds AS (
    SELECT
        DATE_TRUNC(\'day\', created_at) AS day,
        model_name,
        COUNT(*) AS prediction_count,
        AVG(confidence) AS avg_confidence
    FROM {{ ref(\'stg_predictions\') }}
    GROUP BY 1, 2
)
SELECT * FROM daily_preds

---

# models/schema.yml (tests + docs)
models:
  - name: stg_predictions
    description: "Cleaned predictions from raw table"
    columns:
      - name: id
        description: "Primary key"
        tests: [unique, not_null]
      - name: confidence
        tests:
          - not_null
          - accepted_range:
              min_value: 0
              max_value: 1

---

# CLI commands:
# dbt run           execute models
# dbt test          run tests
# dbt docs generate generate docs
# dbt docs serve    serve docs locally
# dbt run --select model_name+  run model and downstream
'''
print(dbt_example)


# models/staging/stg_predictions.sql
{{ config(materialized="view") }}

SELECT
    id,
    user_id,
    model_name,
    prediction,
    confidence::FLOAT AS confidence,
    created_at::TIMESTAMP AS created_at
FROM {{ source('raw', 'predictions') }}
WHERE created_at IS NOT NULL

---

# models/marts/model_performance.sql
{{ config(materialized="table") }}

WITH daily_preds AS (
    SELECT
        DATE_TRUNC('day', created_at) AS day,
        model_name,
        COUNT(*) AS prediction_count,
        AVG(confidence) AS avg_confidence
    FROM {{ ref('stg_predictions') }}
    GROUP BY 1, 2
)
SELECT * FROM daily_preds

---

# models/schema.yml (tests + docs)
models:
  - name: stg_predictions
    description: "Cleaned predictions from raw table"
    columns:
      - name: id
        description: "Primary key"
        tests: [unique, not_null]
      - name: confidence
        tests:
          - not_null
          - accepted_range:
              min_value: 0
              max_value: 1

---

# 

## Delta Lake ACID Transactions on Data Lakes

In [4]:
delta_example = '''
# pip install delta-spark
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Write Delta table
df.write.format("delta").mode("overwrite").save("/data/predictions_delta")

# Time travel query historical versions
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/data/predictions_delta")
df_yesterday = spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("/data/predictions_delta")

# Merge (upsert) update existing rows, insert new ones
dt = DeltaTable.forPath(spark, "/data/predictions_delta")
dt.alias("target").merge(
    new_data.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

# Vacuum remove old files
dt.vacuum(retentionHours=168)  # keep 7 days of history

# Show history
dt.history().show()
'''
print(delta_example)


# pip install delta-spark
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

builder = SparkSession.builder     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Write Delta table
df.write.format("delta").mode("overwrite").save("/data/predictions_delta")

# Time travel query historical versions
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("/data/predictions_delta")
df_yesterday = spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("/data/predictions_delta")

# Merge (upsert) update existing rows, insert new ones
dt = DeltaTable.forPath(spark, "/data/predictions_delta")
dt.alias("target").merge(
    new_data.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll()  .whenNotMatchedInsertAll()  .execute()

## Airflow Pipeline Orchestration

In [5]:
airflow_example = '''
# pip install apache-airflow
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.providers.amazon.aws.operators.s3 import S3CreateObjectOperator
from datetime import datetime, timedelta

default_args = {
    "owner": "data-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
    "email": ["alerts@company.com"]
}

with DAG(
    dag_id="ml_training_pipeline",
    schedule_interval="@daily",
    start_date=datetime(2024, 1, 1),
    catchup=False,
    default_args=default_args,
    tags=["ml", "training"]
) as dag:
    
    # Task 1: Extract features from data warehouse
    extract_features = PythonOperator(
        task_id="extract_features",
        python_callable=lambda: print("Extracting features..."),
    )
    
    # Task 2: Validate data quality
    validate_data = BashOperator(
        task_id="validate_data",
        bash_command="python validate.py --date={{ ds }}",  # Jinja templating
    )
    
    # Task 3: Train model
    def train_model(**context):
        run_date = context["ds"]
        print(f"Training model for date: {run_date}")
        # mlflow.start_run() ...
    
    train = PythonOperator(
        task_id="train_model",
        python_callable=train_model,
        provide_context=True
    )
    
    # Task 4: Evaluate
    evaluate = PythonOperator(
        task_id="evaluate_model",
        python_callable=lambda: print("Evaluating..."),
    )
    
    # Task 5: Deploy if metrics pass
    deploy = BashOperator(
        task_id="deploy_model",
        bash_command="python deploy.py --env=prod",
    )
    
    # Dependencies: linear pipeline
    extract_features >> validate_data >> train >> evaluate >> deploy
'''
print(airflow_example)


# pip install apache-airflow
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.providers.amazon.aws.operators.s3 import S3CreateObjectOperator
from datetime import datetime, timedelta

default_args = {
    "owner": "data-team",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
    "email": ["alerts@company.com"]
}

with DAG(
    dag_id="ml_training_pipeline",
    schedule_interval="@daily",
    start_date=datetime(2024, 1, 1),
    catchup=False,
    default_args=default_args,
    tags=["ml", "training"]
) as dag:

    # Task 1: Extract features from data warehouse
    extract_features = PythonOperator(
        task_id="extract_features",
        python_callable=lambda: print("Extracting features..."),
    )

    # Task 2: Validate data quality
    validate_data = BashOperator(
        task_id="validate_data",
        bash_command="python validate.py --date={{ d

## Great Expectations Data Quality

In [6]:
# pip install great-expectations pandas
import pandas as pd
import numpy as np

# Simulate ML training data
np.random.seed(42)
data = pd.DataFrame({
    "user_id": range(1, 101),
    "age": np.random.randint(18, 80, 100),
    "income": np.random.exponential(50000, 100),
    "label": np.random.choice([0, 1], 100),
    "category": np.random.choice(["A", "B", "C"], 100)
})

# Manual data validation (Great Expectations approach)
def validate_training_data(df):
    errors = []
    
    # Check no nulls in critical columns
    for col in ["user_id", "age", "label"]:
        if df[col].isnull().any():
            errors.append(f"Column {col} has nulls")
    
    # Check value ranges
    if not (df["age"].between(0, 150).all()):
        errors.append("Age out of range [0, 150]")
    if not (df["label"].isin([0, 1]).all()):
        errors.append("Label must be 0 or 1")
    
    # Check uniqueness
    if df["user_id"].duplicated().any():
        errors.append("user_id contains duplicates")
    
    # Check class balance
    class_ratio = df["label"].mean()
    if not (0.1 <= class_ratio <= 0.9):
        errors.append(f"Class imbalance: {class_ratio:.2%} positive")
    
    return errors

errors = validate_training_data(data)
if errors:
    print("Data quality issues:", errors)
else:
    print("Data validation passed!")
    print(f"Dataset: {len(data)} rows, {data.dtypes.to_dict()}")
    print(f"Label distribution: {data['label'].value_counts().to_dict()}")

Data validation passed!
Dataset: 100 rows, {'user_id': dtype('int64'), 'age': dtype('int64'), 'income': dtype('float64'), 'label': dtype('int64'), 'category': dtype('O')}
Label distribution: {1: 53, 0: 47}


## Data Engineering Stack Summary

| Tool | Category | Use Case |
|------|----------|----------|
| Apache Spark | Processing | Large-scale batch transformations |
| Apache Kafka | Streaming | Real-time event pipelines |
| Apache Flink | Streaming | Stateful stream processing |
| dbt | Transform | SQL-based modeling, testing |
| Airflow | Orchestration | Scheduled DAG pipelines |
| Prefect | Orchestration | Python-native, modern UI |
| Delta Lake | Storage | ACID on data lakes |
| Apache Iceberg | Storage | Open table format |
| Great Expectations | Quality | Data validation |
| Fivetran/Airbyte | Ingestion | ELT connectors |
| dbt | Transform | SQL analytics engineering |

## Additional Learning Resources

### Papers
- [Spark: Cluster Computing with Working Sets](https://people.csail.mit.edu/matei/papers/2010/hotcloud_spark.pdf) Zaharia et al.
- [Kafka: A Distributed Messaging System for Log Processing](https://notes.stephenholiday.com/Kafka.pdf) LinkedIn
- [Delta Lake: High-Performance ACID Table Storage](https://www.vldb.org/pvldb/vol13/p3411-armbrust.pdf) Databricks

### Documentation
- [PySpark Docs](https://spark.apache.org/docs/latest/api/python/) Apache Spark Python API
- [Kafka Docs](https://kafka.apache.org/documentation/) Apache Kafka reference
- [dbt Docs](https://docs.getdbt.com/) dbt analytics engineering
- [Airflow Docs](https://airflow.apache.org/docs/) Apache Airflow guide
- [Prefect Docs](https://docs.prefect.io/) Modern Python orchestration
- [Delta Lake Docs](https://docs.delta.io/latest/index.html) ACID data lake

### Books
- [Fundamentals of Data Engineering](https://www.oreilly.com/library/view/fundamentals-of-data/9781098108298/) Joe Reis, Matt Housley
- [Designing Data-Intensive Applications](https://dataintensive.net/) Martin Kleppmann
- [Learning Spark, 2nd ed.](https://www.oreilly.com/library/view/learning-spark-2nd/9781492050032/) O'Reilly
- [Kafka: The Definitive Guide](https://www.oreilly.com/library/view/kafka-the-definitive/9781492043072/) O'Reilly

### Courses
- [Data Engineering Zoomcamp](https://github.com/DataTalksClub/data-engineering-zoomcamp) Free, hands-on
- [DataTalks.Club DE Course](https://datatalks.club/blog/data-engineering-zoomcamp.html) Kafka, Spark, dbt, Airflow